In [1]:
# === Imports ===
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

from sklearn.metrics import mean_squared_error

# === Chargement des données ===
csv_path = r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale.csv"
df = pd.read_csv(csv_path, sep=';', encoding='utf-8-sig')

# Sélection des features numériques uniquement
features = ["Nombre de Titres", "Echéance", "Taux"]
X = df[features]
y = df["Montant"]

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [2]:
# === XGBoost ===
model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_estimators=200, max_depth=5)

# 1️⃣ RMSE initial
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE initial XGBoost : {rmse_base}")

# 2️⃣ RMSE après normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model.fit(X_train_scaled, y_train)
y_pred_scaled = model.predict(X_test_scaled)
rmse_scaled = np.sqrt(mean_squared_error(y_test, y_pred_scaled))
print(f"RMSE après normalisation XGBoost : {rmse_scaled}")

# 3️⃣ Feature Selection
selector = SelectKBest(score_func=f_regression, k='all')
X_train_fs = selector.fit_transform(X_train_scaled, y_train)
X_test_fs = selector.transform(X_test_scaled)
model.fit(X_train_fs, y_train)
y_pred_fs = model.predict(X_test_fs)
rmse_fs = np.sqrt(mean_squared_error(y_test, y_pred_fs))
print(f"RMSE après feature selection XGBoost : {rmse_fs}")

# 4️⃣ Hyperparameter tuning (GridSearchCV)
param_grid = {
    'n_estimators':[200,400,600],
    'max_depth':[3,5,7],
    'learning_rate':[0.01,0.05,0.1],
    'subsample':[0.7,0.8,1.0],
    'colsample_bytree':[0.7,0.8,1.0]
}
grid = GridSearchCV(estimator=xgb.XGBRegressor(objective='reg:squarederror', random_state=42),
                    param_grid=param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid.fit(X_train_scaled, y_train)
best_model = grid.best_estimator_
y_pred_tuned = best_model.predict(X_test_scaled)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
print(f"RMSE après hyperparameter tuning XGBoost : {rmse_tuned}")


RMSE initial XGBoost : 3.8966965450431
RMSE après normalisation XGBoost : 3.8966965450431
RMSE après feature selection XGBoost : 3.8966965450431
RMSE après hyperparameter tuning XGBoost : 3.90079325552296


In [5]:
# === Linear Regression ===
model = LinearRegression()

# 1️⃣ RMSE initial
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE initial Linear Regression : {rmse_base}")

# 2️⃣ RMSE après normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model.fit(X_train_scaled, y_train)
y_pred_scaled = model.predict(X_test_scaled)
rmse_scaled = np.sqrt(mean_squared_error(y_test, y_pred_scaled))
print(f"RMSE après normalisation Linear Regression : {rmse_scaled}")

# 3️⃣ Feature Selection
selector = SelectKBest(score_func=f_regression, k='all')
X_train_fs = selector.fit_transform(X_train_scaled, y_train)
X_test_fs = selector.transform(X_test_scaled)
model.fit(X_train_fs, y_train)
y_pred_fs = model.predict(X_test_fs)
rmse_fs = np.sqrt(mean_squared_error(y_test, y_pred_fs))
print(f"RMSE après feature selection Linear Regression : {rmse_fs}")

# 4️⃣ Hyperparameter tuning Ridge
param_grid = {'alpha':[0.01,0.1,1,10,100]}
grid = GridSearchCV(Ridge(), param_grid, cv=3, scoring='neg_root_mean_squared_error')
grid.fit(X_train_scaled, y_train)
best_model = grid.best_estimator_
y_pred_tuned = best_model.predict(X_test_scaled)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
print(f"RMSE après hyperparameter tuning Linear Regression (Ridge) : {rmse_tuned}")


RMSE initial Linear Regression : 6.687866079408175
RMSE après normalisation Linear Regression : 6.68786607940819
RMSE après feature selection Linear Regression : 6.68786607940819
RMSE après hyperparameter tuning Linear Regression (Ridge) : 6.685798897517014


In [ ]:
# === Random Forest ===
model = RandomForestRegressor(random_state=42, n_estimators=200, max_depth=None)

# 1️⃣ RMSE initial
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE initial Random Forest : {rmse_base}")

# 2️⃣ RMSE après normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model.fit(X_train_scaled, y_train)
y_pred_scaled = model.predict(X_test_scaled)
rmse_scaled = np.sqrt(mean_squared_error(y_test, y_pred_scaled))
print(f"RMSE après normalisation Random Forest : {rmse_scaled}")

# 3️⃣ Feature Selection
selector = SelectKBest(score_func=f_regression, k='all')
X_train_fs = selector.fit_transform(X_train_scaled, y_train)
X_test_fs = selector.transform(X_test_scaled)
model.fit(X_train_fs, y_train)
y_pred_fs = model.predict(X_test_fs)
rmse_fs = np.sqrt(mean_squared_error(y_test, y_pred_fs))
print(f"RMSE après feature selection Random Forest : {rmse_fs}")

# 4️⃣ Hyperparameter tuning
param_grid = {
    'n_estimators':[200,400,600],
    'max_depth':[None,5,10],
    'max_features':['sqrt','log2']
}
grid = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid.fit(X_train_scaled, y_train)
best_model = grid.best_estimator_
y_pred_tuned = best_model.predict(X_test_scaled)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
print(f"RMSE après hyperparameter tuning Random Forest : {rmse_tuned}")


RMSE initial Random Forest : 3.642068565396579
RMSE après normalisation Random Forest : 3.6325479665013427
RMSE après feature selection Random Forest : 3.6325479665013427


In [5]:
# === Gradient Boosting ===
model = GradientBoostingRegressor(random_state=42, n_estimators=200, learning_rate=0.1, max_depth=3)

# 1️⃣ RMSE initial
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE initial Gradient Boosting : {rmse_base}")

# 2️⃣ RMSE après normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model.fit(X_train_scaled, y_train)
y_pred_scaled = model.predict(X_test_scaled)
rmse_scaled = np.sqrt(mean_squared_error(y_test, y_pred_scaled))
print(f"RMSE après normalisation Gradient Boosting : {rmse_scaled}")

# 3️⃣ Feature Selection
selector = SelectKBest(score_func=f_regression, k='all')
X_train_fs = selector.fit_transform(X_train_scaled, y_train)
X_test_fs = selector.transform(X_test_scaled)
model.fit(X_train_fs, y_train)
y_pred_fs = model.predict(X_test_fs)
rmse_fs = np.sqrt(mean_squared_error(y_test, y_pred_fs))
print(f"RMSE après feature selection Gradient Boosting : {rmse_fs}")

# 4️⃣ Hyperparameter tuning
param_grid = {
    'n_estimators':[200,400,600],
    'learning_rate':[0.01,0.05,0.1],
    'max_depth':[3,5,7],
    'subsample':[0.7,0.8,1.0]
}
grid = GridSearchCV(GradientBoostingRegressor(random_state=42), param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid.fit(X_train_scaled, y_train)
best_model = grid.best_estimator_
y_pred_tuned = best_model.predict(X_test_scaled)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
print(f"RMSE après hyperparameter tuning Gradient Boosting : {rmse_tuned}")


RMSE initial Gradient Boosting : 0.08021096328706695
RMSE après normalisation Gradient Boosting : 0.08021082333186975
RMSE après feature selection Gradient Boosting : 0.08021082333186975
RMSE après hyperparameter tuning Gradient Boosting : 0.041433875528404725


In [6]:
# === Lasso ===
model = Lasso(random_state=42)

# 1️⃣ RMSE initial
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE initial Lasso : {rmse_base}")

# 2️⃣ RMSE après normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model.fit(X_train_scaled, y_train)
y_pred_scaled = model.predict(X_test_scaled)
rmse_scaled = np.sqrt(mean_squared_error(y_test, y_pred_scaled))
print(f"RMSE après normalisation Lasso : {rmse_scaled}")

# 3️⃣ Feature Selection
selector = SelectKBest(score_func=f_regression, k='all')
X_train_fs = selector.fit_transform(X_train_scaled, y_train)
X_test_fs = selector.transform(X_test_scaled)
model.fit(X_train_fs, y_train)
y_pred_fs = model.predict(X_test_fs)
rmse_fs = np.sqrt(mean_squared_error(y_test, y_pred_fs))
print(f"RMSE après feature selection Lasso : {rmse_fs}")

# 4️⃣ Hyperparameter tuning
param_grid = {'alpha':[0.001,0.01,0.1,1,10]}
grid = GridSearchCV(Lasso(random_state=42), param_grid, cv=3, scoring='neg_root_mean_squared_error')
grid.fit(X_train_scaled, y_train)
best_model = grid.best_estimator_
y_pred_tuned = best_model.predict(X_test_scaled)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
print(f"RMSE après hyperparameter tuning Lasso : {rmse_tuned}")


RMSE initial Lasso : 0.14161359501931164
RMSE après normalisation Lasso : 0.9581943011707359
RMSE après feature selection Lasso : 0.9581943011707359
RMSE après hyperparameter tuning Lasso : 0.0009581943011719671


In [ ]:
# === Cellule 7 : Interaction - saisie des features et choix du modèle pour prédire "Montant" ===
# Cette cellule n'altère pas les précédentes. Elle (re)entraîne les 5 modèles demandés, puis permet
# de saisir les features et de choisir un modèle pour faire une prédiction.
#
# ⚠️ Attention : dans ta config actuelle, "Montant" est à la fois une feature et la target (y = df["Montant"]).
#    C'est une fuite d'information. Je respecte ton setup sans le modifier, mais garde ce point à l'esprit.

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
import numpy as np

# --- Préparations communes ---
numeric_features = ["Nombre de Titres", "Echéance", "Taux"]
scaler = StandardScaler()

# Pipelines simples (avec normalisation quand demandé)
pipe_lr_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

pipe_rf_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(random_state=42, n_estimators=200, max_depth=None))
])

# --- XGBoost (tuning) ---
xgb_param_grid = {
    'model__n_estimators': [200, 400, 600],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0]
}
pipe_xgb_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("model", xgb.XGBRegressor(objective='reg:squarederror', random_state=42))
])
grid_xgb = GridSearchCV(
    estimator=pipe_xgb_scaled,
    param_grid=xgb_param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
grid_xgb.fit(X_train, y_train)
best_xgb = grid_xgb.best_estimator_

# --- Gradient Boosting (tuning) ---
gb_param_grid = {
    'model__n_estimators': [200, 400, 600],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__max_depth': [3, 5, 7],
    'model__subsample': [0.7, 0.8, 1.0]
}
from sklearn.ensemble import GradientBoostingRegressor
pipe_gb_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("model", GradientBoostingRegressor(random_state=42))
])
grid_gb = GridSearchCV(
    estimator=pipe_gb_scaled,
    param_grid=gb_param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
grid_gb.fit(X_train, y_train)
best_gb = grid_gb.best_estimator_

# --- Lasso (tuning) ---
lasso_param_grid = {'model__alpha': [0.001, 0.01, 0.1, 1, 10]}
pipe_lasso_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(random_state=42, max_iter=10000))
])
grid_lasso = GridSearchCV(
    estimator=pipe_lasso_scaled,
    param_grid=lasso_param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
grid_lasso.fit(X_train, y_train)
best_lasso = grid_lasso.best_estimator_

# --- Linear Regression (après normalisation) ---
pipe_lr_scaled.fit(X_train, y_train)

# --- Random Forest (après normalisation) ---
pipe_rf_scaled.fit(X_train, y_train)

# --- Dictionnaire des modèles disponibles + calcul RMSE test pour info ---
def _rmse_test(model):
    y_pred = model.predict(X_test)
    return float(np.sqrt(mean_squared_error(y_test, y_pred)))

model_zoo = {
    1: ("XGBoost (tuning) - RMSE le plus faible du grid", best_xgb, _rmse_test(best_xgb)),
    2: ("Linear Regression (après normalisation)",       pipe_lr_scaled, _rmse_test(pipe_lr_scaled)),
    3: ("Random Forest (après normalisation)",           pipe_rf_scaled, _rmse_test(pipe_rf_scaled)),
    4: ("Gradient Boosting (tuning)",                    best_gb, _rmse_test(best_gb)),
    5: ("Lasso (tuning)",                                best_lasso, _rmse_test(best_lasso)),
}

# --- Saisie utilisateur & prédiction ---
def _parse_number(s):
    """Convertit une chaîne en float, accepte virgule ou point."""
    if isinstance(s, (int, float)):
        return float(s)
    s = str(s).strip().replace(',', '.')
    return float(s)

def saisir_features_et_predire():
    print("\n=== Prédiction du Montant ===")
    print("Saisir les valeurs pour les features suivantes dans l'ordre :")
    print(" -> " + ", ".join(numeric_features))
    print("   (Astuce: tu peux utiliser la virgule ',' pour les décimales)")

    # Saisie des 4 features dans l'ordre défini par 'features'
    x_vals = []
    for f in numeric_features:
        while True:
            try:
                val = _parse_number(input(f"  {f} = "))
                x_vals.append(val)
                break
            except ValueError:
                print("  Valeur invalide. Réessaye (nombre attendu).")

    # Choix du modèle
    print("\nChoisis un modèle :")
    for k, (label, _, rmse) in model_zoo.items():
        print(f"  {k}. {label}   |  RMSE(test) ≈ {rmse:.6f}")

    choice = None
    while choice not in model_zoo:
        try:
            choice = int(input("Ton choix (1-5) = ").strip())
        except ValueError:
            pass

    label, model, rmse = model_zoo[choice]
    x_array = np.array(x_vals, dtype=float).reshape(1, -1)
    y_hat = float(model.predict(x_array)[0])

    print("\n=== Résultat ===")
    print(f"Modèle choisi : {label}")
    print(f"RMSE (jeu de test) ≈ {rmse:.6f}")
    print(f"Prédiction du Montant pour l'entrée fournie : {y_hat:.6f}")

# Lance la fonction interactive après exécution de la cellule, si tu le souhaites :
saisir_features_et_predire()



=== Prédiction du Montant ===
Saisir les valeurs pour les features suivantes dans l'ordre :
 -> Nombre de Titres, Montant, Echéance, Taux
   (Astuce: tu peux utiliser la virgule ',' pour les décimales)
  Valeur invalide. Réessaye (nombre attendu).
  Valeur invalide. Réessaye (nombre attendu).
  Valeur invalide. Réessaye (nombre attendu).
  Valeur invalide. Réessaye (nombre attendu).
  Valeur invalide. Réessaye (nombre attendu).
